# Практика 2 — качество данных

## Подготовка

In [1]:
from pathlib import Path
import pandas as pd
from pandas.testing import assert_frame_equal
DATA = Path("../data")
LOCAL = Path("../data")
WORK = Path("../results")
WORK.mkdir(exist_ok=True)
print("pandas", pd.__version__)

pandas 3.1.0


## Два входных файла

In [2]:
decisions = pd.read_csv(
    DATA / "decisions.csv", dtype={"lei": "string", "county_code": "string"}
)
export = pd.read_csv(LOCAL / "training_export.csv", dtype="string")
raw_copy = export.copy(deep=True)
print(decisions.shape, export.shape)
print(export.dtypes)
export.head(5)

(6830, 7) (6832, 7)
source_row       string
activity_year    string
lei              string
county_code      string
action_taken     string
loan_amount      string
income           string
dtype: object


,source_row,activity_year,lei,county_code,action_taken,loan_amount,income
0,2,2023,549300VZVN841I2ILS84,11001,1,oops,72.0
1,3,2023,549300P1347BFCBHJV60,11001,88,585000.0,174.0
2,5,2023,549300P1347BFCBHJV60,11001,1,275000.0,BAD
3,7,2023,549300RPOGWJRH63HS39,11001,1,295000.0,63.0
4,8,2023,549300RPOGWJRH63HS39,11001,1,1125000.0,247.0


## Профиль входа

In [3]:
print(export.isna().sum())
print(export["action_taken"].value_counts(dropna=False))

source_row         0
activity_year      0
lei                0
county_code        0
action_taken       0
loan_amount        0
income           342
dtype: int64
action_taken
1     5957
3      691
2      183
88       1
Name: count, dtype: Int64


## A. Проверки входа

In [4]:
# Проверка обязательных столбцов
required_columns = [
    "source_row", "activity_year", "lei",
    "county_code", "action_taken", "loan_amount", "income"
]

missing_columns = set(required_columns) - set(export.columns)
assert not missing_columns, f"Не хватает столбцов: {missing_columns}"

# Повторяющиеся номера исходных строк
source_row_text = export["source_row"].str.strip()
duplicate_source_row = source_row_text.duplicated(keep=False)

print("Повторяющиеся source_row:")
display(export.loc[duplicate_source_row])

# Проверка допустимых исходов
action_text = export["action_taken"].str.strip()
invalid_action = ~action_text.isin(["1", "2", "3"])

print("Недопустимые action_taken:")
display(export.loc[invalid_action, ["source_row", "action_taken"]])

# Частоты кодов территории, включая пропуски
county_text = export["county_code"].str.strip()
print("Частоты county_code:")
display(county_text.replace("", pd.NA).value_counts(dropna=False))

Повторяющиеся source_row:


,source_row,activity_year,lei,county_code,action_taken,loan_amount,income
10,15,2023,549300RPOGWJRH63HS39,11001,1,945000.0,270.0
11,16,2023,549300RPOGWJRH63HS39,11001,1,305000.0,77.0
6830,15,2023,549300RPOGWJRH63HS39,11001,1,945000.0,270.0
6831,16,2023,549300RPOGWJRH63HS39,11001,1,305000.0,77.0


Недопустимые action_taken:


,source_row,action_taken
1,3,88


Частоты county_code:


county_code
11001    6832
Name: count, dtype: Int64

## Преобразование суммы

In [5]:
work = export.copy(deep=True)
work["county_code"] = work["county_code"].str.strip()
for field in ["source_row", "loan_amount", "action_taken"]:
    numbers = pd.to_numeric(work[field], errors="coerce")
    failed = work[field].notna() & numbers.isna()
    print(field, int(failed.sum()))
    print(work.loc[failed, ["source_row", field]])
    work[field] = numbers

source_row 0
Empty DataFrame
Columns: [source_row, source_row]
Index: []
loan_amount 1
   source_row loan_amount
0           2        oops
action_taken 0
Empty DataFrame
Columns: [source_row, action_taken]
Index: []


## B. Ошибки преобразования дохода

In [6]:
income_numbers = pd.to_numeric(
    export["income"].astype("string").str.strip(),
    errors="coerce"
)

income_parse_error = (
    export["income"].astype("string").str.strip().notna()
    & export["income"].astype("string").str.strip().ne("")
    & income_numbers.isna()
)

original_income_missing = (
    export["income"].isna()
    | export["income"].astype("string").str.strip().eq("")
)

print("Новые ошибки преобразования дохода:",
      int(income_parse_error.sum()))

print("Исходные пропуски дохода:",
      int(original_income_missing.sum()))

print("Пропуски после преобразования:",
      int(income_numbers.isna().sum()))

print("Исходные значения с ошибками:")
display(export.loc[income_parse_error, ["source_row", "income"]])

# Проверяем, что исходная выгрузка осталась неизменной
assert_frame_equal(export, raw_copy)
print("Проверка пройдена: export не изменён.")

Новые ошибки преобразования дохода: 1
Исходные пропуски дохода: 342
Пропуски после преобразования: 343
Исходные значения с ошибками:


,source_row,income
2,5,BAD


Проверка пройдена: export не изменён.


## Проверка повторной доставки

In [7]:
known_key = export["source_row"].notna()
repeated = export.loc[known_key].drop_duplicates()
assert repeated["source_row"].is_unique, "Конфликтующие версии ключа"
replay_mask = known_key & export["source_row"].duplicated()
unique = work.loc[~replay_mask].copy()
print(len(export), len(unique), int(replay_mask.sum()))

6832 6830 2


## C. Строки с нарушениями обязательных правил

In [8]:
# source_row должен быть известен, чтобы можно было проверить доставку
known_row = work["source_row"].notna()

# Для одного source_row проверяем, не противоречат ли друг другу записи
row_versions = work.loc[known_row].groupby("source_row").nunique(
    dropna=False
)

conflicting_rows = row_versions.index[row_versions.gt(1).any(axis=1)]

print("Номера строк с конфликтующими версиями:")
display(work.loc[work["source_row"].isin(conflicting_rows)])

Номера строк с конфликтующими версиями:


,source_row,activity_year,lei,county_code,action_taken,loan_amount,income


## Функция по разобранным шагам

In [9]:
def clean_export(source):
    out = source.copy(deep=True)
    known_key = out["source_row"].notna()
    repeated = out.loc[known_key].drop_duplicates()
    assert repeated["source_row"].is_unique, "Конфликтующие версии ключа"
    replay_mask = known_key & out["source_row"].duplicated()
    replay_n = int(replay_mask.sum())
    out = out.loc[~replay_mask].copy()
    out["county_code"] = out["county_code"].str.strip()
    parse_counts = {}
    for field in ["source_row", "loan_amount", "income", "action_taken"]:
        numbers = pd.to_numeric(out[field], errors="coerce")
        parse_counts[field] = int((out[field].notna() & numbers.isna()).sum())
        out[field] = numbers
    reason = pd.Series("", index=out.index, dtype="string")
    reason.loc[out["source_row"].isna()] += "unknown_source_row;"
    bad_amount = out["loan_amount"].isna() | out["loan_amount"].le(0)
    reason.loc[bad_amount] += "invalid_loan_amount;"
    reason.loc[~out["action_taken"].isin([1, 2, 3])] += "invalid_action_taken;"
    bad_county = ~out["county_code"].eq("11001").fillna(False)
    reason.loc[bad_county] += "invalid_county;"
    invalid = reason.ne("")
    clean = out.loc[~invalid].copy()
    quarantine = source.loc[out.index[invalid]].copy()
    quarantine["reason"] = reason.loc[invalid]
    audit = {
        "input": len(source), "replays": replay_n,
        "clean": len(clean), "quarantine": len(quarantine),
        "parse_errors": parse_counts,
    }
    assert audit["input"] == audit["replays"] + audit["clean"] + audit["quarantine"]
    return clean, quarantine, audit

## Результат функции

In [10]:
result, rejected, audit = clean_export(export)
print(audit)
rejected

{'input': 6832, 'replays': 2, 'clean': 6828, 'quarantine': 2, 'parse_errors': {'source_row': 0, 'loan_amount': 1, 'income': 1, 'action_taken': 0}}


,source_row,activity_year,lei,county_code,action_taken,loan_amount,income,reason
0,2,2023,549300VZVN841I2ILS84,11001,1,oops,72.0,invalid_loan_amount;
1,3,2023,549300P1347BFCBHJV60,11001,88,585000.0,174.0,invalid_action_taken;


## D. Сохранность и повторяемость

In [11]:
def approval_stats(df, label):
    actions = pd.to_numeric(df["action_taken"], errors="coerce")
    valid = actions.isin([1, 2, 3])

    denominator = int(valid.sum())
    numerator = int(actions.isin([1, 2]).sum())

    rate = numerator / denominator if denominator else float("nan")

    return {
        "group": label,
        "approved_1_or_2": numerator,
        "denominator_actions_1_to_3": denominator,
        "approval_share": rate,
        "approval_percent": rate * 100
    }


# Для оригинала приводим action_taken к числу
original_for_stats = decisions.copy()
original_for_stats["action_taken"] = pd.to_numeric(
    original_for_stats["action_taken"], errors="coerce"
)

# Для подготовленного набора income уже преобразован
known_income = result.loc[result["income"].notna()].copy()

comparison = pd.DataFrame([
    approval_stats(original_for_stats, "Оригинал decisions"),
    approval_stats(result, "Подготовленная result"),
    approval_stats(known_income, "result с известным income")
])

display(comparison)

,group,approved_1_or_2,denominator_actions_1_to_3,approval_share,approval_percent
0,Оригинал decisions,6139,6830,0.898829,89.882870
1,Подготовленная result,6137,6828,0.898799,89.879906
2,result с известным income,5859,6485,0.903470,90.346955


## Показатель и выбранная группа

In [12]:
def show_approval(label, table):
    n = len(table)
    assert n > 0
    k = table["action_taken"].isin([1, 2]).sum()
    print(label, k, n, f"{100 * k / n:.2f}%")

show_approval("Оригинал", decisions)
show_approval("После очистки", result)
complete = result.dropna(subset=["income"])
show_approval("Известный доход", complete)

Оригинал 6139 6830 89.88%
После очистки 6137 6828 89.88%
Известный доход 5859 6485 90.35%


## E. Сохранение и отчёт

In [14]:
print("Повторы source_row:")
print(export["source_row"].duplicated().sum())

print("\nНедопустимые action_taken:")
print(export.loc[~export["action_taken"].isin(["1", "2", "3"]), "action_taken"].value_counts(dropna=False))

print("\nРаспределение county_code:")
print(export["county_code"].value_counts(dropna=False))

Повторы source_row:
2

Недопустимые action_taken:
action_taken
88    1
Name: count, dtype: Int64

Распределение county_code:
county_code
11001      6831
 11001        1
Name: count, dtype: Int64


In [15]:
income_numbers = pd.to_numeric(export["income"], errors="coerce")

income_parse_error = export["income"].notna() & income_numbers.isna()

print("Исходные пропуски в income:", export["income"].isna().sum())
print("Новые ошибки преобразования:", income_parse_error.sum())
print("Пропуски после преобразования:", income_numbers.isna().sum())
print("Исходная таблица не изменена:", export.equals(raw_copy))

Исходные пропуски в income: 342
Новые ошибки преобразования: 1
Пропуски после преобразования: 343
Исходная таблица не изменена: True


In [17]:
import pandas as pd

# 1. Количество строк и баланс
input_rows = len(export)
unique_rows = len(unique)
clean_rows = len(result)
quarantine_rows = len(rejected)

# Сколько строк убрали как повторные доставки
duplicate_deliveries = input_rows - unique_rows

print("=== C. Карантин и баланс ===")
print("Исходных строк:", input_rows)
print("Уникальных записей после обработки повторов:", unique_rows)
print("Повторных доставок:", duplicate_deliveries)
print("Записей в карантине:", quarantine_rows)
print("Записей в очищенном наборе:", clean_rows)

print(
    "Баланс выполняется:",
    input_rows == clean_rows + quarantine_rows + duplicate_deliveries
)

# 2. Проверки качества
result_again, rejected_again, audit_again = clean_export(export)

print("\n=== D. Проверки качества ===")
print(
    "Повторный запуск даёт те же результаты:",
    result.equals(result_again)
    and rejected.equals(rejected_again)
    and audit == audit_again
)
print("Исходная таблица не изменилась:", export.equals(raw_copy))
print(
    "source_row уникален:",
    not result["source_row"].duplicated().any()
)
print(
    "Только допустимые action_taken:",
    result["action_taken"].astype(str).isin(["1", "2", "3"]).all()
)

# Укажи правильное название столбца с суммой кредита,
# если в твоём ноутбуке оно отличается.
amount_col = "loan_amount"

if amount_col in result.columns:
    amounts = pd.to_numeric(result[amount_col], errors="coerce")
    print(
        "Сумма кредита известна и положительна:",
        amounts.notna().all() and amounts.gt(0).all()
    )
else:
    print(
        f"Проверка суммы кредита: столбец {amount_col!r} не найден."
    )

print(
    "Для каждой записи карантина указана причина:",
    "reason" in rejected.columns
    and rejected["reason"].notna().all()
    and rejected["reason"].astype(str).str.strip().ne("").all()
)

print(
    "Баланс строк выполняется:",
    input_rows == clean_rows + quarantine_rows + duplicate_deliveries
)

print(
    "Пропусков income после очистки:",
    result["income"].isna().sum()
)

# 3. Процент одобрений
# Код 1 означает одобрение.
def approval_stats(df, name):
    actions = df["action_taken"].astype(str).str.strip()
    total = len(df)
    approved = int(actions.eq("1").sum())
    percent = approved / total * 100 if total else 0

    return {
        "Таблица и группа": name,
        "Одобрения": approved,
        "Все записи": total,
        "Процент": round(percent, 2)
    }

# Оригинал
original_stats = approval_stats(decisions, "Оригинал")

# После учебной очистки
clean_stats = approval_stats(result, "После учебной очистки")

# Очищенные записи с известным числовым доходом
income_numeric = pd.to_numeric(result["income"], errors="coerce")
known_income = result.loc[income_numeric.notna()].copy()
known_income_stats = approval_stats(
    known_income, "Известный доход после очистки"
)

approval_table = pd.DataFrame([
    original_stats,
    clean_stats,
    known_income_stats
])

print("\n=== Показатель: доля одобрений ===")
display(approval_table)

=== C. Карантин и баланс ===
Исходных строк: 6832
Уникальных записей после обработки повторов: 6830
Повторных доставок: 2
Записей в карантине: 2
Записей в очищенном наборе: 6828
Баланс выполняется: True

=== D. Проверки качества ===
Повторный запуск даёт те же результаты: True
Исходная таблица не изменилась: True
source_row уникален: True
Только допустимые action_taken: True
Сумма кредита известна и положительна: True
Для каждой записи карантина указана причина: True
Баланс строк выполняется: True
Пропусков income после очистки: 343

=== Показатель: доля одобрений ===


,Таблица и группа,Одобрения,Все записи,Процент
0,Оригинал,5956,6830,87.20
1,После учебной очистки,5954,6828,87.20
2,Известный доход после очистки,5709,6485,88.03


In [13]:
from pathlib import Path

RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

result.to_csv(
    RESULTS_DIR / "clean.csv",
    index=False,
    encoding="utf-8"
)

rejected.to_csv(
    RESULTS_DIR / "quarantine.csv",
    index=False,
    encoding="utf-8"
)

import json

with open(RESULTS_DIR / "quality_report.json", "w", encoding="utf-8") as f:
    json.dump(audit, f, ensure_ascii=False, indent=2, default=str)